# Oxfordshire Children's Home & Care Home Planning Scrapers

Scrapes planning applications for **children's homes** and **care homes** from 5 councils:
1. Oxford City Council
2. Cherwell District Council
3. West Oxfordshire District Council
4. Vale of White Horse District Council
5. South Oxfordshire District Council

**Output:** One CSV per council, plus a combined CSV.

## 0. Install Dependencies

In [ ]:
# Run once to install required packages
import subprocess, sys
packages = ['selenium', 'webdriver-manager', 'beautifulsoup4', 'requests', 'pandas', 'lxml']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed.')

## 1. Shared Configuration & Utilities

In [ ]:
import time
import re
import os
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime, date

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_DIR = 'planning_outputs'
HEADLESS   = False   # Keep False — visible browser bypasses most bot detection
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Search terms → canonical category mapping ─────────────────────────────────
# Each entry: (search_term_to_submit, canonical_category)
SEARCH_CONFIG = [
    # Care homes only (children's homes removed per user request)
    ("care home",                "care home"),
    ("residential care home",    "care home"),
    ("nursing home",             "care home"),
]
IDOX_SEARCH_TERMS  = [s for s, _ in SEARCH_CONFIG]
AGILE_SEARCH_TERMS = IDOX_SEARCH_TERMS

def categorise(text):
    """
    Classify free text into 'children\'s home', 'care home', or 'other'.
    Children\'s home is checked first to avoid misclassifying
    'children\'s residential care home' as a generic care home.
    """
    t = str(text).lower()
    childrens_patterns = [
        r"children.s\s+(residential\s+)?home",
        r"residential\s+children.s",
        r"children.s\s+residential\s+care",
        r"residential\s+care\s+for\s+children",
        r"young\s+people.s\s+(residential\s+)?home",
        r"childrens\s+home",
    ]
    for p in childrens_patterns:
        if re.search(p, t):
            return "children's home"
    care_patterns = [
        r"care\s+home",
        r"nursing\s+home",
        r"residential\s+care\s+home",
        r"residential\s+care\s+facilit",
    ]
    for p in care_patterns:
        if re.search(p, t):
            return "care home"
    for term, cat in SEARCH_CONFIG:
        if term.lower() in t:
            return cat
    return 'other'

# ── Column schema ─────────────────────────────────────────────────────────────
COLS = [
    'council', 'reference', 'date_received', 'date_validated',
    'address', 'description', 'applicant', 'agent',
    'status', 'decision', 'decision_date',
    'category', 'search_term', 'url'
]

# ── Human behaviour helpers ───────────────────────────────────────────────────

def human_delay(mn=2.0, mx=5.0):
    time.sleep(random.uniform(mn, mx))

def short_delay():
    time.sleep(random.uniform(0.5, 1.5))

def micro_delay():
    time.sleep(random.uniform(0.1, 0.45))

def human_type(element, text):
    """Type character-by-character with realistic inter-keystroke timing."""
    element.clear()
    micro_delay()
    for char in text:
        element.send_keys(char)
        time.sleep(random.uniform(0.05, 0.20))
    micro_delay()

def human_scroll(driver):
    """Scroll down and slightly back up, like a human skimming results."""
    driver.execute_script(f'window.scrollBy(0, {random.randint(250, 650)});')
    micro_delay()
    driver.execute_script(f'window.scrollBy(0, -{random.randint(40, 130)});')
    micro_delay()

def human_move_and_click(driver, element):
    """Hover over element briefly then click, like a real user."""
    try:
        actions = ActionChains(driver)
        actions.move_to_element_with_offset(
            element, random.randint(-4, 4), random.randint(-3, 3)
        )
        actions.pause(random.uniform(0.15, 0.45))
        actions.click()
        actions.perform()
    except Exception:
        # JS click fallback for elements ActionChains can't interact with
        try:
            driver.execute_script("arguments[0].click();", element)
        except Exception:
            element.click()

def check_blocked(driver):
    """
    Detect genuine rate-limit / block pages.
    Uses PHRASE PAIRS so single innocent words (like 'automated' in a footer)
    don't trigger false positives. Only fires when multiple signals appear together.
    """
    page = driver.page_source.lower()

    # Hard indicators — these alone are enough
    hard_indicators = [
        'too many requests',
        'rate limit exceeded',
        'you have been blocked',
        'your ip has been',
        'access has been denied',
        'automated access',           # phrase, not just 'automated'
        'automated requests',
        'looks like you are using automated',
    ]
    for phrase in hard_indicators:
        if phrase in page:
            print(f'\n  \u26a0\ufe0f  Block detected ("{phrase}") — waiting 90s...')
            time.sleep(90)
            driver.refresh()
            human_delay(4, 8)
            # Re-check after refresh
            if any(p in driver.page_source.lower() for p in hard_indicators):
                print('  \u26a0\ufe0f  Still blocked — waiting 120s more...')
                time.sleep(120)
                driver.refresh()
                human_delay(5, 10)
            return True

    # Soft indicators — only block if the page also looks wrong (very short or no planning content)
    soft_indicators = ['captcha', 'robot check', 'ddos', 'cloudflare']
    page_has_planning = any(w in page for w in ['application', 'planning', 'council', 'decision'])
    for phrase in soft_indicators:
        if phrase in page and not page_has_planning:
            print(f'\n  \u26a0\ufe0f  Possible block ("{phrase}", no planning content) — waiting 60s...')
            time.sleep(60)
            driver.refresh()
            human_delay(3, 6)
            return True

    return False

def make_driver(headless=HEADLESS):
    """Create a Chrome WebDriver with anti-detection hardening."""
    opts = Options()
    if headless:
        opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    w, h = random.randint(1280, 1600), random.randint(800, 960)
    opts.add_argument(f'--window-size={w},{h}')
    opts.add_argument(
        'user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    # Hide navigator.webdriver flag
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {'source': '''
        Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
        Object.defineProperty(navigator, 'plugins',   { get: () => [1,2,3,4,5] });
        Object.defineProperty(navigator, 'languages', { get: () => ['en-GB','en'] });
        window.chrome = { runtime: {} };
    '''})
    driver.implicitly_wait(8)
    return driver

def empty_row(council, search_term, url=''):
    row = {c: '' for c in COLS}
    row['council']     = council
    row['search_term'] = search_term
    row['category']    = categorise(search_term)
    row['url']         = url
    return row

def finalise_category(rec):
    """Re-derive category from description text once we have it."""
    desc = rec.get('description', '')
    if desc:
        cat = categorise(desc)
        if cat != 'other':
            rec['category'] = cat
    return rec

def save_csv(records, filename):
    records = [finalise_category(r) for r in records]
    df = pd.DataFrame(records, columns=COLS)
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'  \u2192 Saved {len(df)} rows to {path}')
    print(f'     Categories: {df["category"].value_counts().to_dict()}')
    return df

print('Configuration loaded \u2713')

---
## Scrapers 1–3: Idox Planning Portal (Oxford City, Cherwell, West Oxfordshire)

These three councils run the **Idox Public Access** planning portal.
- Search the **keyword/description field** for each term
- Paginate through all results
- Click each application to get full details (applicant, agent, decision)

**If search fails:** Run the diagnostic cell first to see what fields are actually on the page.

In [ ]:
# ── DIAGNOSTIC: uncomment & run if any Idox scraper fails ────────────────────

def diagnose_idox_page(url):
    """Print all form inputs and forms — run this to debug field detection."""
    print(f'Opening: {url}')
    driver = make_driver(headless=False)
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'lxml')
        print('\n--- ALL INPUT FIELDS ---')
        for inp in soup.find_all(['input','textarea','select']):
            print(f"  tag={inp.name}  id={inp.get('id','—')!r:35s}  name={inp.get('name','—')!r:45s}  type={inp.get('type','')!r}  display={'none' if 'display:none' in (inp.get('style','') or '') else 'visible'}")
        print('\n--- FORMS (with their inputs) ---')
        for fi, form in enumerate(soup.find_all('form')):
            print(f"  Form {fi}: action={form.get('action','')!r}  id={form.get('id','')!r}")
            for inp in form.find_all(['input','textarea','select']):
                print(f"    {inp.name}  id={inp.get('id','')!r:30s}  name={inp.get('name','')!r:40s}  type={inp.get('type','')!r}")
        print(f'\nURL: {driver.current_url}')
    finally:
        input('Press ENTER to close browser...')
        driver.quit()


def diagnose_cherwell_submit():
    """
    Special diagnostic: types a search term into Cherwell's keyword box,
    then shows exactly what URL/page results after each submit method.
    Run this to find out why the search returns no results.
    """
    url = 'https://planningregister.cherwell.gov.uk/'
    driver = make_driver(headless=False)
    try:
        driver.get(url)
        time.sleep(3)
        print(f'Page title: {driver.title}')
        print(f'URL: {driver.current_url}')

        # Find and display all forms
        soup = BeautifulSoup(driver.page_source, 'lxml')
        forms = soup.find_all('form')
        print(f'\n{len(forms)} form(s) on page:')
        for fi, form in enumerate(forms):
            inputs = form.find_all(['input','button'])
            print(f'  Form {fi}: action={form.get("action","")!r}')
            for inp in inputs:
                print(f'    {inp.name}  id={inp.get("id","")!r}  name={inp.get("name","")!r}  type={inp.get("type","")!r}  value={inp.get("value","")!r}')

        # Find the keyword field using our function
        print('\nLocating keyword field...')
        field = find_idox_search_field(driver, 'Cherwell')
        fid = field.get_attribute('id')
        fname = field.get_attribute('name')
        print(f'Found: id={fid!r} name={fname!r}')
        print(f'Interactable: {is_interactable(field)}')

        # Type into it
        human_move_and_click(driver, field)
        time.sleep(0.5)
        human_type(field, "care home")
        print('Typed "care home" into field')
        time.sleep(1)

        # Find which form this field belongs to — then find that form\'s submit
        print('\nLooking for submit button in the same form as the keyword field...')
        soup2 = BeautifulSoup(driver.page_source, 'lxml')
        kw_inp = soup2.find('input', id=fid) or soup2.find('input', attrs={'name': fname})
        if kw_inp:
            parent_form = kw_inp.find_parent('form')
            if parent_form:
                print(f'  Keyword field is inside form: action={parent_form.get("action","")!r}')
                submits = parent_form.find_all(['input','button'], type=re.compile('submit|button', re.I))
                print(f'  Submit buttons in that form:')
                for s in submits:
                    print(f'    {s.name}  id={s.get("id","")!r}  value={s.get("value","")!r}')
            else:
                print('  WARNING: keyword field is NOT inside a <form> tag')

        input('\nInspect the browser manually. Press ENTER when ready to try submitting...')

        # Try pressing Enter on the field (most reliable)
        from selenium.webdriver.common.keys import Keys
        field2 = driver.find_element(By.ID, fid) if fid else driver.find_element(By.NAME, fname)
        field2.send_keys(Keys.RETURN)
        time.sleep(3)
        print(f'After ENTER — URL: {driver.current_url}')
        print(f'Page contains "results": {"result" in driver.page_source.lower()}')
        print(f'Page contains "no result": {"no result" in driver.page_source.lower()}')
        input('\nPress ENTER to close...')

    finally:
        driver.quit()


# ── Uncomment to run diagnostics ──────────────────────────────────────────────
# diagnose_idox_page('https://planningregister.cherwell.gov.uk/')
# diagnose_cherwell_submit()   # ← run this first to debug Cherwell submit

# ── Idox portal scraper ───────────────────────────────────────────────────────

# Standard Idox Public Access portals (Oxford, West Oxon)
PUBLICACCESS_COUNCILS = {
    'Oxford City':      'https://public.oxford.gov.uk/online-applications/search.do?action=simple&searchType=Application',
    'West Oxfordshire': 'https://publicaccess.westoxon.gov.uk/online-applications/search.do?action=simple&searchType=Application',
}

# Cherwell uses a different Idox-based portal with different HTML structure
CHERWELL_URL = 'https://planningregister.cherwell.gov.uk/'

# Legacy alias so other code doesn't break
IDOX_COUNCILS = {**PUBLICACCESS_COUNCILS, 'Cherwell': CHERWELL_URL}

# Confirmed field ID from diagnostic — same across all 3 Idox portals
IDOX_SEARCH_FIELD_ID = 'simpleSearchString'

# Per-council overrides if the above ever stops working:
# 'Oxford City': ('id', 'someOtherId')   or  ('name', 'someOtherName')
IDOX_FIELD_OVERRIDES = {
    'Oxford City':      None,
    'Cherwell':         None,
    'West Oxfordshire': None,
}


def dismiss_cookies(driver):
    selectors = [
        (By.CSS_SELECTOR, '#cookieAccept'),
        (By.CSS_SELECTOR, '#accept-cookies'),
        (By.CSS_SELECTOR, '.cookie-accept'),
        (By.XPATH, "//button[contains(translate(text(),'ACCEPT','accept'),'accept')]"),
        (By.XPATH, "//a[contains(translate(text(),'ACCEPT','accept'),'accept all')]"),
        (By.XPATH, "//input[@value='Accept' or @value='accept']"),
    ]
    for by, sel in selectors:
        try:
            el = WebDriverWait(driver, 2).until(EC.element_to_be_clickable((by, sel)))
            el.click()
            short_delay()
            return True
        except Exception:
            pass
    return False


def is_interactable(el):
    """Return True if element is visible and has non-zero size."""
    try:
        size = el.size
        return el.is_displayed() and size['width'] > 0 and size['height'] > 0
    except Exception:
        return False


def find_idox_search_field(driver, council_name):
    """
    Locate the keyword/description search field.
    Explicitly skips the application-reference field and hidden elements.
    For portals with two search boxes (like Cherwell homepage), targets the
    one labelled 'keyword' or 'description', NOT 'reference' or 'application number'.
    """
    override = IDOX_FIELD_OVERRIDES.get(council_name)
    if override:
        by_str, val = override
        by = {'id': By.ID, 'name': By.NAME, 'css': By.CSS_SELECTOR}[by_str]
        el = driver.find_element(by, val)
        if is_interactable(el):
            return el

    # XPath strategies that specifically target keyword/description fields
    # and explicitly exclude reference number fields
    keyword_xpaths = [
        # Input whose id/name contains 'simpleSearch' (confirmed Oxford/WestOxon)
        "//input[contains(@id,'simpleSearch') or contains(@name,'simpleSearch')]",
        # Input following a label that says 'keyword' or 'description' (case-insensitive)
        "//label[contains(translate(normalize-space(.),'KEYWORDINSGABTPOC','keywordinsgabtpoc'),'keyword')]"
            "/following::input[@type='text' or not(@type)][1]",
        "//label[contains(translate(normalize-space(.),'DESCRIPTIONKYWOR','descriptionkywor'),'description')]"
            "/following::input[@type='text' or not(@type)][1]",
        # Input whose id/name contains 'keyword' (not 'reference')
        "//input[contains(@id,'eyword') or contains(@name,'eyword')]",
        # Input whose id/name contains 'description' but NOT 'reference'
        "//input[(contains(@id,'escription') or contains(@name,'escription'))"
            " and not(contains(@id,'ref') or contains(@name,'ref'))]",
    ]
    for xpath in keyword_xpaths:
        try:
            candidates = driver.find_elements(By.XPATH, xpath)
            for el in candidates:
                if is_interactable(el):
                    fid = el.get_attribute('id') or ''
                    fname = el.get_attribute('name') or ''
                    # Double-check it's not a reference field
                    skip = ['reference', 'appnum', 'appno', 'appref']
                    if any(s in fid.lower() or s in fname.lower() for s in skip):
                        continue
                    print(f'    \u2713 Keyword field: id={fid!r} name={fname!r}')
                    return el
        except Exception:
            pass

    # Fallback: walk ALL visible text inputs, skip reference/date/postcode fields
    print('    Auto-detect: scanning all visible text inputs...')
    skip_patterns = ['reference','appnum','appno','appref','postcode',
                     'date','year','ward','parish','agent','submit']
    all_inputs = driver.find_elements(By.XPATH, "//input[@type='text' or not(@type)]")
    candidates = []
    for inp in all_inputs:
        if not is_interactable(inp):
            continue
        fid   = (inp.get_attribute('id')   or '').lower()
        fname = (inp.get_attribute('name') or '').lower()
        fplace = (inp.get_attribute('placeholder') or '').lower()
        if any(k in fid or k in fname for k in skip_patterns):
            continue
        # Score by how much it looks like a keyword/description field
        score = sum([
            3 if 'keyword'     in fid   or 'keyword'     in fname  else 0,
            3 if 'description' in fid   or 'description' in fname  else 0,
            2 if 'search'      in fid   or 'search'      in fname  else 0,
            1 if 'keyword'     in fplace or 'description' in fplace else 0,
        ])
        candidates.append((score, inp, fid, fname))
        print(f'    Candidate: id={fid!r} name={fname!r} score={score}')

    if candidates:
        candidates.sort(key=lambda x: -x[0])  # highest score first
        score, el, fid, fname = candidates[0]
        print(f'    \u2713 Best candidate: id={fid!r} name={fname!r} score={score}')
        return el

    raise NoSuchElementException(
        f'Cannot find keyword field on {council_name}. '
        'Run diagnose_idox_page() then add an entry to IDOX_FIELD_OVERRIDES.'
    )


def idox_submit_search(driver, field=None):
    """
    Submit the search. Preferred method is pressing Enter on the search field
    itself — this is the most reliable cross-portal approach and matches how
    a real user would submit (especially on Cherwell which has multiple forms).
    """
    from selenium.webdriver.common.keys import Keys
    # Method 1: Press Enter on the field (most reliable — works even with multiple forms)
    if field is not None:
        try:
            field.send_keys(Keys.RETURN)
            return
        except Exception:
            pass

    # Method 2: Find the submit button that is in the same form as a visible text input
    try:
        soup = BeautifulSoup(driver.page_source, 'lxml')
        all_text_inputs = driver.find_elements(
            By.XPATH, "//input[@type='text' or not(@type)]"
        )
        for inp_el in all_text_inputs:
            if not is_interactable(inp_el):
                continue
            inp_id   = inp_el.get_attribute('id')   or ''
            inp_name = inp_el.get_attribute('name') or ''
            skip = ['reference','appnum','appno','appref']
            if any(s in inp_id.lower() or s in inp_name.lower() for s in skip):
                continue
            # Found a likely keyword field — find its parent form's submit
            kw_soup = soup.find('input', id=inp_id) or soup.find('input', attrs={'name': inp_name})
            if kw_soup:
                parent_form = kw_soup.find_parent('form')
                if parent_form:
                    form_id     = parent_form.get('id', '')
                    form_action = parent_form.get('action', '')
                    # Find the submit button within this specific form
                    form_xpath = (
                        f"//form[@id='{form_id}']//input[@type='submit']"
                        if form_id else
                        f"//form[@action='{form_action}']//input[@type='submit']"
                    )
                    try:
                        btn = driver.find_element(By.XPATH, form_xpath)
                        human_move_and_click(driver, btn)
                        return
                    except NoSuchElementException:
                        pass
    except Exception:
        pass

    # Method 3: Generic submit button fallback
    for by, sel in [
        (By.CSS_SELECTOR, 'input[type="submit"][value="Search"]'),
        (By.CSS_SELECTOR, 'button[type="submit"]'),
        (By.CSS_SELECTOR, 'input[type="submit"]'),
    ]:
        try:
            btn = driver.find_element(by, sel)
            if is_interactable(btn):
                human_move_and_click(driver, btn)
                return
        except NoSuchElementException:
            pass

    # Last resort: JS submit
    driver.execute_script("document.querySelector('form').submit();")


def idox_scrape_council(council_name, base_url, headless=HEADLESS, portal_type='publicaccess'):
    """Scrape one Idox council for all search terms."""
    all_records = []
    seen_refs   = set()   # deduplicate across search terms
    driver      = make_driver(headless)
    try:
        for term in IDOX_SEARCH_TERMS:
            print(f'  Searching "{term}" on {council_name}...')
            records = idox_search_term(driver, council_name, base_url, term, portal_type=portal_type)
            # Deduplicate: skip if we already have this reference
            new = [r for r in records if r.get('reference') not in seen_refs or not r.get('reference')]
            seen_refs.update(r['reference'] for r in new if r.get('reference'))
            print(f'    {len(records)} found, {len(new)} new after dedup')
            all_records.extend(new)
            human_delay(3, 7)   # inter-search pause
    finally:
        driver.quit()
    return all_records


def idox_search_term(driver, council_name, base_url, term, portal_type='publicaccess'):
    """
    Two-phase scrape for one search term:
      Phase 1 - paginate through results, collecting application URLs only
      Phase 2 - visit each URL directly (never use driver.back() — it causes
                Cherwell to re-render and re-find the Next button infinitely)
    """
    from urllib.parse import urljoin
    records = []

    # ── Submit search ─────────────────────────────────────────────────────────
    driver.get(base_url)
    human_delay(2, 4)
    check_blocked(driver)
    dismiss_cookies(driver)
    human_scroll(driver)
    short_delay()

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, 'input'))
        )
    except TimeoutException:
        print('    WARNING: inputs slow to load')

    try:
        field = find_idox_search_field(driver, council_name)
    except NoSuchElementException as e:
        print(f'    ERROR finding field: {e}')
        return records

    human_move_and_click(driver, field)
    short_delay()
    human_type(field, term)
    human_delay(0.8, 2.0)
    idox_submit_search(driver, field=field)
    human_delay(2, 5)
    check_blocked(driver)

    page_text = driver.page_source.lower()
    if 'no results' in page_text or 'returned no results' in page_text:
        print(f'    No results for "{term}"')
        return records

    # ── Phase 1: collect ALL application URLs across all pages ────────────────
    all_app_urls     = []
    seen_urls        = set()
    seen_page_hashes = set()
    page_num = 1

    while True:
        print(f'    Page {page_num} (collecting)...', end=' ', flush=True)
        check_blocked(driver)
        soup = BeautifulSoup(driver.page_source, 'lxml')

        # Application links on Idox are inside <li class="searchresult"> elements
        # Each li has one <a> whose href contains keyVal= or applicationDetails
        result_links = []

        # Use Selenium directly (not BeautifulSoup) so JS-rendered content is visible.
        # Wait briefly for results to appear after navigation.
        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    'a[href*="/Planning/Display/"], a[href*="keyVal"], a[href*="applicationDetails"], li.searchresult a'))
            )
        except TimeoutException:
            pass  # no results on this page

        # Strategy 1: Cherwell-style /Planning/Display/ links
        for el in driver.find_elements(By.CSS_SELECTOR, 'a[href*="/Planning/Display/"]'):
            href = el.get_attribute('href') or ''
            if href:
                result_links.append(href)

        # Strategy 2: Standard Idox keyVal links (Oxford, West Oxon)
        if not result_links:
            for el in driver.find_elements(By.CSS_SELECTOR,
                    'a[href*="keyVal"], a[href*="applicationDetails"], li.searchresult a'):
                href = el.get_attribute('href') or ''
                if href and href not in result_links:
                    result_links.append(href)

        new_links = [u for u in result_links if u not in seen_urls]

        page_hash = frozenset(result_links)
        if page_hash and page_hash in seen_page_hashes:
            print(f'duplicate page — stopping')
            break
        seen_page_hashes.add(page_hash)

        for u in new_links:
            seen_urls.add(u)
        all_app_urls.extend(new_links)
        print(f'{len(new_links)} new (total: {len(all_app_urls)})')

        # Find Next page button — Cherwell uses <a> with text 'Next' in a pager
        next_btn = None
        for xpath in [
            "//a[normalize-space(.)='Next']",
            "//a[normalize-space(.)='next']",
            "//a[contains(@class,'next') and not(contains(@class,'disabled'))]",
            "//li[contains(@class,'next') and not(contains(@class,'disabled'))]/a",
            "//a[@rel='next']",
            "//a[normalize-space(.)='>']",
            "//nav[contains(@id,'pager') or contains(@class,'pager')]//a[last()]",
        ]:
            try:
                btn = driver.find_element(By.XPATH, xpath)
                if btn.is_displayed() and btn.is_enabled():
                    next_btn = btn
                    break
            except NoSuchElementException:
                pass

        if next_btn:
            human_move_and_click(driver, next_btn)
            page_num += 1
            human_delay(2, 4)
        else:
            break

    print(f'    → {len(all_app_urls)} unique application URLs found')

    # ── Phase 2: visit each detail URL directly ───────────────────────────────
    print(f'    Visiting {len(all_app_urls)} detail pages...')
    for idx, app_url in enumerate(all_app_urls):
        rec = idox_get_application_detail(driver, app_url, council_name, term, base_url)
        records.append(rec)
        if idx > 0 and idx % random.randint(5, 10) == 0:
            human_delay(4, 9)
        else:
            human_delay(1.5, 3.5)
        # No driver.back() — visiting next URL directly avoids portal re-render loop

    return records


def parse_idox_summary_row(row, council, term, base_url):
    rec = empty_row(council, term, base_url)
    for attr, patterns in [
        ('reference',   ['refval', 'reference', 'appnum']),
        ('address',     ['address', 'location']),
        ('description', ['description', 'proposal', 'summary']),
        ('status',      ['status', 'decision']),
    ]:
        for p in patterns:
            tag = row.find(class_=re.compile(p, re.I))
            if tag:
                rec[attr] = tag.get_text(strip=True)
                break
    return rec


def publicaccess_get_detail(driver, app_url, council, term, base_url):
    """
    Parse a standard Idox Public Access detail page (Oxford City, West Oxon).
    These use classic <th>Label</th><td>Value</td> table pairs.
    """
    from urllib.parse import urljoin
    rec = empty_row(council, term, app_url)
    try:
        driver.get(app_url)
        human_delay(1.5, 3)
        check_blocked(driver)
        human_scroll(driver)
        soup = BeautifulSoup(driver.page_source, 'lxml')

        def get_field(soup_obj, *labels):
            for label in labels:
                for el in soup_obj.find_all(['th', 'dt', 'td', 'label']):
                    if label.lower() in el.get_text(strip=True).lower():
                        sib = el.find_next_sibling(['td', 'dd'])
                        if sib and sib.get_text(strip=True):
                            return sib.get_text(strip=True)
                        parent = el.parent
                        if parent:
                            tds = parent.find_all(['td', 'dd'])
                            for i, td in enumerate(tds):
                                if label.lower() in td.get_text().lower() and i + 1 < len(tds):
                                    v = tds[i + 1].get_text(strip=True)
                                    if v:
                                        return v
            return ''

        # Reference
        for sel in ['span#referenceNumber', '#appNumber', 'h1', '.pageTitle']:
            el = soup.select_one(sel)
            if el:
                txt = el.get_text(strip=True)
                if re.search(r'\d{2}/\d{4}|[A-Z]{2}/\d+', txt):
                    rec['reference'] = txt
                    break
        if not rec['reference']:
            rec['reference'] = get_field(soup, 'Reference', 'App No', 'Application No')

        rec['address']        = get_field(soup, 'Address', 'Location', 'Site Address')
        rec['description']    = get_field(soup, 'Proposal', 'Description', 'Development')
        rec['date_received']  = get_field(soup, 'Received', 'Date Received', 'Registered')
        rec['date_validated'] = get_field(soup, 'Validated', 'Valid Date', 'Date Valid')
        rec['status']         = get_field(soup, 'Status', 'Application Status', 'Current Status')
        rec['decision']       = get_field(soup, 'Decision', 'Delegated Decision', 'Outcome')
        rec['decision_date']  = get_field(soup, 'Decision Date', 'Date of Decision', 'Decision Issued')
        rec['applicant']      = get_field(soup, 'Applicant', 'Applicant Name')
        rec['agent']          = get_field(soup, 'Agent', "Agent's Name", "Agents's Name")

        # Contacts tab fallback (some Public Access portals need a separate page)
        if not rec['applicant']:
            contacts_url = None
            for a in soup.find_all('a', href=True):
                href = a['href']
                link_text = a.get_text(strip=True).lower()
                if re.search(r'contact|applicant|agent', link_text) and \
                   re.search(r'(keyVal|applicationDetails|Display|applicant)', href, re.I) and \
                   href != '#':
                    contacts_url = href if href.startswith('http') else urljoin(base_url, href)
                    break
            if contacts_url:
                driver.get(contacts_url)
                human_delay(1.5, 3)
                soup2 = BeautifulSoup(driver.page_source, 'lxml')
                rec['applicant'] = get_field(soup2, 'Applicant', 'Applicant Name')
                rec['agent']     = get_field(soup2, 'Agent', "Agent's Name")

        rec = finalise_category(rec)
    except Exception as e:
        print(f'      WARNING detail page error: {e}')
        rec['description'] = rec.get('description', '') or f'ERROR: {e}'
    return rec


def idox_get_application_detail_cherwell(driver, app_url, council, term, base_url):
    """
    Parse a Cherwell (and other Idox) planning application detail page.

    Confirmed Cherwell HTML structure (from diagnostic):
      Each field is a single <td> containing:
        - Label text directly in the <td> (e.g. "Location")
        - <br/>
        - <div class="singlerowsize"> or <div class="twinrowsize">
            <span>VALUE</span>
          </div>

    All tabs (Main Details, Applicant/Agents, etc.) are present in the DOM
    as <div id="Applicant/ Agents" style="display:none"> — no clicking needed,
    just parse the hidden divs directly.
    """
    from urllib.parse import urljoin
    rec = empty_row(council, term, app_url)
    try:
        driver.get(app_url)
        human_delay(1.5, 3)
        check_blocked(driver)
        human_scroll(driver)
        soup = BeautifulSoup(driver.page_source, 'lxml')

        def get_field(soup_obj, *labels):
            """
            Extract value from Cherwell's label+<br/>+<div><span>value</span></div> pattern.
            Each <td> contains the label as direct text, then a div/span with the value.
            We find any <td> whose direct text matches the label, then return the span text.
            """
            for label in labels:
                lab = label.lower().strip()
                for td in soup_obj.find_all('td'):
                    # Get the direct text of the td (excluding child element text)
                    direct_text = ''.join(
                        t for t in td.strings
                        if t.parent == td or t.parent.name == 'br'
                    ).strip().lower()
                    # Match if the direct text starts with or equals the label
                    if direct_text.startswith(lab) or direct_text == lab:
                        # Value is in the first <span> inside a div.twinrowsize or div.singlerowsize
                        span = td.select_one('div.twinrowsize span, div.singlerowsize span, div span')
                        if span:
                            val = span.get_text(' ', strip=True)
                            if val:
                                return val
                        # Fallback: any non-empty text after stripping the label
                        full_text = td.get_text(' ', strip=True)
                        # Remove the label prefix
                        for lbl in labels:
                            if full_text.lower().startswith(lbl.lower()):
                                val = full_text[len(lbl):].strip().lstrip(':- ')
                                if val:
                                    return val
            return ''

        # ── Reference ─────────────────────────────────────────────────────────
        # Cherwell puts it in the page <h1> as "Planning Application - 26/00395/SCOP"
        h1 = soup.select_one('h1')
        if h1:
            txt = h1.get_text(strip=True)
            m = re.search(r'(\d{2}/\d{4,}[/\w]*)', txt)
            if m:
                rec['reference'] = m.group(1)
        if not rec['reference']:
            rec['reference'] = get_field(soup, 'Application Number', 'Reference')

        # ── Main Details tab ──────────────────────────────────────────────────
        # All tabs are in the DOM; select the Main Details div
        main_div = soup.select_one('#tab-content') or soup

        rec['address']        = get_field(main_div, 'Location', 'Address', 'Site Address')
        rec['description']    = get_field(main_div, 'Proposal', 'Description', 'Development')
        rec['date_received']  = get_field(main_div, 'Received Date', 'Date Received', 'Registered')
        rec['date_validated'] = get_field(main_div, 'Valid Date', 'Validated', 'Date Validated')
        rec['status']         = get_field(main_div, 'Status')
        rec['decision']       = get_field(main_div, 'Decision')
        rec['decision_date']  = get_field(main_div, 'Decision Issued Date', 'Decision Date')

        # ── Applicant / Agents tab ────────────────────────────────────────────
        # This tab is present in the DOM but hidden (display:none).
        # We don't need to click it — just parse the div directly.
        agents_div = None
        for div in soup.find_all('div', id=True):
            if 'agent' in div['id'].lower() or 'applicant' in div['id'].lower():
                agents_div = div
                break
        if not agents_div:
            # Fallback: look in whole page
            agents_div = soup

        rec['applicant'] = get_field(agents_div, 'Applicant')
        rec['agent']     = get_field(agents_div, 'Agent')

        # Finalise category from actual description
        rec = finalise_category(rec)

    except Exception as e:
        print(f'      WARNING detail page error: {e}')
        rec['description'] = rec.get('description','') or f'ERROR: {e}'
    return rec


def idox_get_application_detail_standard(driver, app_url, council, term, base_url):
    """
    Parse standard Idox Public Access portal detail pages (Oxford City, West Oxon).
    These use proper <th>label</th><td>value</td> pairs — the original working parser.
    """
    from urllib.parse import urljoin
    rec = empty_row(council, term, app_url)
    try:
        driver.get(app_url)
        human_delay(1.5, 3)
        check_blocked(driver)
        human_scroll(driver)
        soup = BeautifulSoup(driver.page_source, 'lxml')

        def get_field(soup_obj, *labels):
            for label in labels:
                for el in soup_obj.find_all(['th', 'dt', 'td', 'label']):
                    if label.lower() in el.get_text(strip=True).lower():
                        sib = el.find_next_sibling(['td', 'dd'])
                        if sib and sib.get_text(strip=True):
                            return sib.get_text(strip=True)
                        parent = el.parent
                        if parent:
                            tds = parent.find_all(['td', 'dd'])
                            for i, td in enumerate(tds):
                                if label.lower() in td.get_text().lower() and i + 1 < len(tds):
                                    v = tds[i + 1].get_text(strip=True)
                                    if v:
                                        return v
            return ''

        # Reference
        for sel in ['span#referenceNumber', '#appNumber', 'h1', '.pageTitle']:
            el = soup.select_one(sel)
            if el:
                txt = el.get_text(strip=True)
                if re.search(r'\d{2}/\d{4}|[A-Z]{2}/\d+', txt):
                    rec['reference'] = txt
                    break
        if not rec['reference']:
            rec['reference'] = get_field(soup, 'Reference', 'App No', 'Application No')

        rec['address']        = get_field(soup, 'Address', 'Location', 'Site Address')
        rec['description']    = get_field(soup, 'Proposal', 'Description', 'Development')
        rec['date_received']  = get_field(soup, 'Received', 'Date Received', 'Registered')
        rec['date_validated'] = get_field(soup, 'Validated', 'Valid Date', 'Date Valid')
        rec['status']         = get_field(soup, 'Status', 'Application Status', 'Current Status')
        rec['decision']       = get_field(soup, 'Decision', 'Delegated Decision', 'Outcome')
        rec['decision_date']  = get_field(soup, 'Decision Date', 'Date of Decision', 'Decision Issued')
        rec['applicant']      = get_field(soup, 'Applicant', 'Applicant Name')
        rec['agent']          = get_field(soup, 'Agent', "Agent's Name", "Agents's Name")
        rec = finalise_category(rec)

    except Exception as e:
        print(f'      WARNING detail page error: {e}')
        rec['description'] = rec.get('description', '') or f'ERROR: {e}'
    return rec


def idox_get_application_detail(driver, app_url, council, term, base_url):
    """Route to the correct detail parser based on council."""
    if council == 'Cherwell':
        return idox_get_application_detail_cherwell(driver, app_url, council, term, base_url)
    else:
        return idox_get_application_detail_standard(driver, app_url, council, term, base_url)


print('Idox scraper functions defined \u2713')

### Run Scraper 1 — Oxford City Council

In [ ]:
print('=== Oxford City Council ===')
oxford_records = idox_scrape_council(
    council_name='Oxford City',
    base_url=IDOX_COUNCILS['Oxford City']
)
oxford_df = save_csv(oxford_records, 'oxford_city_planning.csv')
oxford_df.head()

### Run Scraper 2 — Cherwell District Council

In [ ]:
print('=== Cherwell District Council ===')
cherwell_records = idox_scrape_council(
    council_name='Cherwell',
    base_url=IDOX_COUNCILS['Cherwell']
)
cherwell_df = save_csv(cherwell_records, 'cherwell_planning.csv')
cherwell_df.head()

### Run Scraper 3 — West Oxfordshire District Council

In [ ]:
print('=== West Oxfordshire District Council ===')
westoxon_records = idox_scrape_council(
    council_name='West Oxfordshire',
    base_url=IDOX_COUNCILS['West Oxfordshire']
)
westoxon_df = save_csv(westoxon_records, 'west_oxfordshire_planning.csv')
westoxon_df.head()

---
## Scrapers 4 & 5: Agile Applications Portal (Vale of White Horse & South Oxfordshire)

These two councils run the **Agile Applications** portal. The approach:
- Iterate through **every 3-month window** from 1995 to today using URL parameters
- Fetch the results page for each window
- Filter rows whose description contains *children's home* or *care home*
- For matching applications, fetch the detail page to get applicant/agent info

In [ ]:
from datetime import date

try:
    from dateutil.relativedelta import relativedelta
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dateutil', '-q'])
    from dateutil.relativedelta import relativedelta

AGILE_COUNCILS = {
    'Vale of White Horse': 'https://data.whitehorsedc.gov.uk/java/support/Main.jsp',
    'South Oxfordshire':   'https://data.southoxon.gov.uk/ccm/support/Main.jsp',
}

AGILE_START_DATE = date(1995, 1, 1)
AGILE_END_DATE   = date.today()


def build_agile_url(base_url, start_date, end_date):
    return (
        f"{base_url}?MODULE=ApplicationCriteriaList&TYPE=Application"
        f"&PARISH=ALL&AREA=&TXTSEARCH=&APP_TYPE=&APPTYPE=ALL&APP_STATUS="
        f"&SDAY={start_date.day}&SMONTH={start_date.month}&SYEAR={start_date.year}"
        f"&EDAY={end_date.day}&EMONTH={end_date.month}&EYEAR={end_date.year}"
        f"&Submit=Search"
    )


def date_windows(start, end, months=3):
    current = start
    while current < end:
        window_end = min(current + relativedelta(months=months) - relativedelta(days=1), end)
        yield current, window_end
        current = window_end + relativedelta(days=1)


def agile_parse_results_page(page_source, base_url, council_name):
    """
    Parse the Agile results page.

    Structure (confirmed by diagnostic):
      div.tablediv
        div.rowdiv  (header row — skip)
        div.hr
        div.rowdiv  (one per application)
          div.celldiv [0] — reference number + <a href="Main.jsp?MODULE=ApplicationDetails&REF=...">
          div.celldiv [1] — contains two child divs:
                              child[0] = address/location
                              child[1] = proposal/description
          div.celldiv [2] — date registered
        div.hr
        div.rowdiv  (next application)
        ...
    """
    soup = BeautifulSoup(page_source, 'lxml')
    matches = []

    # Find all rowdivs that represent applications (have an <a> with MODULE=ApplicationDetails)
    for rowdiv in soup.find_all('div', class_='rowdiv'):
        celldivs = rowdiv.find_all('div', class_='celldiv', recursive=False)
        if len(celldivs) < 3:
            continue

        # Cell 0: reference number + link
        ref_cell  = celldivs[0]
        link_tag  = ref_cell.find('a', href=True)
        if not link_tag:
            continue
        reference = link_tag.get_text(strip=True)
        href      = link_tag['href']
        detail_url = href if href.startswith('http') else urljoin(base_url, href)

        # Cell 1: address (first child div) + description (second child div)
        loc_cell     = celldivs[1]
        child_divs   = loc_cell.find_all('div', recursive=False)
        address      = child_divs[0].get_text(' ', strip=True) if len(child_divs) > 0 else ''
        description  = child_divs[1].get_text(' ', strip=True) if len(child_divs) > 1 else loc_cell.get_text(' ', strip=True)

        # Cell 2: date registered
        date_received = celldivs[2].get_text(' ', strip=True)

        # Check if this row matches any of our search terms
        row_text = (address + ' ' + description).lower()
        matched_term = None
        for term, _ in SEARCH_CONFIG:
            if term.lower() in row_text:
                matched_term = term
                break
        if not matched_term:
            continue

        rec = empty_row(council_name, matched_term, detail_url)
        rec['reference']    = reference
        rec['date_received'] = date_received
        rec['address']       = address
        rec['description']   = description
        rec = finalise_category(rec)
        matches.append((rec, detail_url))

    return matches


def agile_parse_detail_page(soup, rec):
    """
    Parse an Agile detail page (Vale of White Horse / South Oxon).

    Confirmed structure: pure <div> label/value pairs in document order.
    Pattern:  <div>Label</div>  immediately followed by  <div>Value</div>
    No tables, no th/td. Values may contain newlines (multi-line fields).

    Confirmed labels:
      Description, Location, Applicant, Agent,
      Date Received, Registration Date, Decision, Appeal, Application Type
    """
    KNOWN_LABELS = {
        'description', 'location', 'address', 'site address',
        'applicant', 'applicant name', 'agent', 'agent name',
        'date received', 'registration date', 'target decision date',
        'decision', 'appeal', 'application type', 'application progress',
        'case officer',
    }

    # Collect all leaf divs (no div children) with non-empty text
    leaf_divs = []
    for d in soup.find_all('div'):
        if not d.find('div'):
            txt = d.get_text(' ', strip=True)
            if txt and len(txt) < 600:
                leaf_divs.append(txt)

    # Walk pairs: label div followed immediately by value div
    kv = {}
    for i, txt in enumerate(leaf_divs):
        key = txt.lower().rstrip(':').strip()
        if key in KNOWN_LABELS and i + 1 < len(leaf_divs):
            candidate = leaf_divs[i + 1]
            if candidate.lower().rstrip(':').strip() not in KNOWN_LABELS:
                kv[key] = candidate

    def get(*keys):
        for k in keys:
            v = kv.get(k.lower(), '')
            if v:
                return v
        return ''

    # Description — always overwrite from detail page (list page truncates it)
    desc = get('description')
    if desc:
        rec['description'] = desc

    if not rec.get('address'):
        rec['address'] = get('location', 'address', 'site address')

    if not rec.get('date_received'):
        rec['date_received'] = get('date received')

    rec['date_validated'] = get('registration date')

    # Decision: full string e.g. "Refusal of Planning Permission on 15th December 1997"
    # Split into outcome + date
    decision_full = get('decision')
    if decision_full:
        m = re.search(
            r'\bon\s+(\d{1,2}(?:st|nd|rd|th)?\s+\w+\s+\d{4}|\d{1,2}\s+\w+\s+\d{4})\b',
            decision_full, re.I
        )
        if m:
            rec['decision']      = decision_full[:m.start()].strip()
            rec['decision_date'] = m.group(1)
        else:
            rec['decision'] = decision_full

    # Capture appeal outcome too
    appeal = get('appeal')
    if appeal and 'no appeal' not in appeal.lower():
        if not rec.get('decision'):
            rec['decision'] = appeal
        else:
            rec['decision'] += ' | Appeal: ' + appeal

    # Applicant: first line = name, remaining lines = address
    applicant_raw = get('applicant', 'applicant name')
    if applicant_raw:
        rec['applicant'] = applicant_raw.split('\n')[0].strip()

    agent_raw = get('agent', 'agent name')
    if agent_raw:
        rec['agent'] = agent_raw.split('\n')[0].strip()

    rec = finalise_category(rec)
    return rec

def agile_scrape_council(council_name, base_url, headless=HEADLESS):
    """
    Scrape one Agile portal council across all quarterly windows.
    Uses Selenium (portals return 403 to plain requests).
    """
    all_records = []
    seen_refs   = set()
    windows     = list(date_windows(AGILE_START_DATE, AGILE_END_DATE, months=3))
    print(f'  Scanning {len(windows)} quarterly windows ({AGILE_START_DATE} to {AGILE_END_DATE})...')

    driver = make_driver(headless)
    try:
        for i, (start, end) in enumerate(windows):
            if i % 20 == 0:
                print(f'  Window {i+1}/{len(windows)}: {start} to {end}')

            url = build_agile_url(base_url, start, end)
            driver.get(url)
            human_delay(1.5, 3.0)
            check_blocked(driver)

            # Quick pre-check: skip if none of our terms appear on the page
            page_lower = driver.page_source.lower()
            if not any(t.lower() in page_lower for t, _ in SEARCH_CONFIG):
                continue

            matches = agile_parse_results_page(driver.page_source, base_url, council_name)
            if not matches:
                continue

            print(f'    ✓ {len(matches)} match(es) in {start}–{end}')

            for rec, detail_url in matches:
                ref = rec.get('reference', '')
                if ref and ref in seen_refs:
                    continue
                if ref:
                    seen_refs.add(ref)

                # Visit detail page for applicant/agent/decision
                if detail_url:
                    try:
                        driver.get(detail_url)
                        human_delay(1.5, 3.0)
                        check_blocked(driver)
                        soup = BeautifulSoup(driver.page_source, 'lxml')
                        rec = agile_parse_detail_page(soup, rec)
                        try:
                            driver.back()
                            human_delay(1.0, 2.0)
                        except Exception as nav_err:
                            print(f'      Session lost ({type(nav_err).__name__}) — reconnecting...')
                            try:
                                driver.quit()
                            except Exception:
                                pass
                            driver = make_driver(headless)
                            driver.get(url)  # return to current window URL
                            human_delay(3, 5)
                    except Exception as e:
                        print(f'      Detail error for {ref}: {e}')

                all_records.append(rec)

            print(f'    Running total: {len(all_records)} records')

            # Longer break every 40 windows
            if i > 0 and i % 40 == 0:
                print('    Taking a longer break...')
                human_delay(8, 15)
            else:
                human_delay(0.8, 2.0)

    finally:
        driver.quit()

    return all_records


print('Agile scraper functions defined ✓')

In [ ]:
# ── Test new detail parser on known Cherwell URLs ────────────────────────────
def test_cherwell_detail(url):
    driver = make_driver(headless=False)
    try:
        rec = idox_get_application_detail(driver, url, 'Cherwell', 'care home',
                                          IDOX_COUNCILS['Cherwell'])
        print(f'URL: {url}')
        for k, v in rec.items():
            if k != 'url':
                print(f'  {k:20s}: {repr(str(v)[:80])}')
    finally:
        driver.quit()

# Test 1: recent application with known data
test_cherwell_detail('https://planningregister.cherwell.gov.uk/Planning/Display/26/00395/SCOP')
print()
# Test 2: older application that should have a decision
test_cherwell_detail('https://planningregister.cherwell.gov.uk/Planning/Display/26/00350/F')


### Run Scraper 4 — Vale of White Horse District Council

In [ ]:
print('=== Vale of White Horse District Council ===')
whitehorse_records = agile_scrape_council(
    council_name='Vale of White Horse',
    base_url=AGILE_COUNCILS['Vale of White Horse']
)
whitehorse_df = save_csv(whitehorse_records, 'vale_of_white_horse_planning.csv')
whitehorse_df.head()

### Run Scraper 5 — South Oxfordshire District Council

In [ ]:
print('=== South Oxfordshire District Council ===')
southoxon_records = agile_scrape_council(
    council_name='South Oxfordshire',
    base_url=AGILE_COUNCILS['South Oxfordshire']
)
southoxon_df = save_csv(southoxon_records, 'south_oxfordshire_planning.csv')
southoxon_df.head()

---
## Combine All Results

In [ ]:
# Combine all councils into one master CSV
all_dfs = []
for name, fname in [
    ('Oxford City',         'oxford_city_planning.csv'),
    ('Cherwell',            'cherwell_planning.csv'),
    ('West Oxfordshire',    'west_oxfordshire_planning.csv'),
    ('Vale of White Horse', 'vale_of_white_horse_planning.csv'),
    ('South Oxfordshire',   'south_oxfordshire_planning.csv'),
]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        all_dfs.append(df)
        print(f'  {name}: {len(df)} rows')

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    # Deduplicate on reference + council
    combined = combined.drop_duplicates(subset=['council', 'reference'], keep='first')
    combined_path = os.path.join(OUTPUT_DIR, 'ALL_councils_planning.csv')
    combined.to_csv(combined_path, index=False)
    print(f'\n✓ Combined CSV: {len(combined)} unique applications → {combined_path}')

    # Summary breakdown
    print('\nBreakdown by council and search term:')
    print(combined.groupby(['council', 'search_term']).size().to_string())
else:
    print('No data found. Run the individual scrapers above first.')

---
## Quick Analysis: Private vs Local Authority Applicants

In [ ]:
# Heuristic classification of applicant type
# (only works where applicant name was successfully scraped)

LOCAL_AUTH_KEYWORDS = [
    'county council', 'district council', 'city council', 'borough council',
    'oxfordshire county', 'oxford city', 'cherwell', 'west oxon', 'whitehorse',
    'south oxon', 'nhs', 'trust', 'department for', 'ministry', 'gov.uk'
]
PRIVATE_KEYWORDS = [
    'ltd', 'limited', 'plc', 'llp', 'inc', 'group', 'care homes', 'care group',
    'properties', 'developments', 'healthcare'
]

def classify_applicant(name):
    if not name or str(name).strip() == '':
        return 'Unknown'
    n = str(name).lower()
    if any(k in n for k in LOCAL_AUTH_KEYWORDS):
        return 'Local Authority / Public Body'
    if any(k in n for k in PRIVATE_KEYWORDS):
        return 'Private Company'
    return 'Individual / Other'

try:
    combined['applicant_type'] = combined['applicant'].apply(classify_applicant)
    print('Applicant type breakdown:')
    print(combined['applicant_type'].value_counts().to_string())
    print()
    print('Breakdown by council + applicant type:')
    print(combined.groupby(['council', 'applicant_type']).size().to_string())
    combined.to_csv(os.path.join(OUTPUT_DIR, 'ALL_councils_planning.csv'), index=False)
    print('\n✓ Combined CSV updated with applicant_type column.')
except NameError:
    print('Run the combine cell above first.')

---
## Troubleshooting Notes

### If Oxford / Cherwell / West Oxon scrapers fail:
- Set `HEADLESS = False` in Cell 2 so the browser is visible — this often bypasses bot detection
- Some Idox portals redirect to a login page when headless; visible browser usually works
- If the search field ID differs, inspect the page and update `'description'` in `idox_search_term()`

### If Vale of White Horse / South Oxon scrapers are slow:
- They scan ~120 quarterly windows back to 1995 — expected run time is 5–20 minutes
- Reduce `PAGE_DELAY` to `0.5` if you want faster (less polite) scraping
- Change `AGILE_START_DATE` to e.g. `date(2010, 1, 1)` to limit the scan

### Applicant field empty for many rows:
- Applicant name requires visiting each detail page; some councils hide this behind a login
- The `applicant_type` classifier in the last cell will label these as `Unknown`
- You can manually review the `url` column links for those rows